In [ ]:
import os
import json

# 配置你的根目录名称
BASE_DIR = "template"
OUTPUT_JSON = "experiment_config.json"

# 定义映射关系，确保 JSON 结构整齐
PRICE_MAP = {
    "budget-priced 价格低": "budget",
    "medium-priced 价格中": "medium",
    "premium-priced 价格高": "premium"
}

FORMATS = ["FAQ", "List", "Paragraph"]

def build_config():
    config = {
        "Apparel": {"budget": {}, "medium": {}, "premium": {}},
        "Cosmetics": {"budget": {}, "medium": {}, "premium": {}},
        "Electronics": {"budget": {}, "medium": {}, "premium": {}}
    }

    # 遍历品类
    for cat_folder in os.listdir(BASE_DIR):
        cat_key = ""
        if "Apparel" in cat_folder: cat_key = "Apparel"
        elif "Cosmetics" in cat_folder: cat_key = "Cosmetics"
        elif "Electronics" in cat_folder: cat_key = "Electronics"
        else: continue

        cat_path = os.path.join(BASE_DIR, cat_folder)
        if not os.path.isdir(cat_path): continue

        # 遍历价位
        for price_folder in os.listdir(cat_path):
            price_key = ""
            for k, v in PRICE_MAP.items():
                if k in price_folder:
                    price_key = v
                    break
            if not price_key: continue

            price_path = os.path.join(cat_path, price_folder)
            
            # 找到具体品牌文件夹（如 Champion...）
            for brand_folder in os.listdir(price_path):
                brand_path = os.path.join(price_path, brand_folder)
                if not os.path.isdir(brand_path): continue

                # 处理中文：核心卖点前置
                cn_path = os.path.join(brand_path, "核心卖点前置")
                if os.path.exists(cn_path):
                    if "CN" not in config[cat_key][price_key]: config[cat_key][price_key]["CN"] = {}
                    for f in os.listdir(cn_path):
                        for fmt in FORMATS:
                            if fmt in f:
                                with open(os.path.join(cn_path, f), 'r', encoding='utf-8') as file:
                                    config[cat_key][price_key]["CN"][fmt] = file.read()

                # 处理英文：Core Selling Points Highlighted First
                en_path = os.path.join(brand_path, "Core Selling Points Highlighted First")
                if os.path.exists(en_path):
                    if "EN" not in config[cat_key][price_key]: config[cat_key][price_key]["EN"] = {}
                    for f in os.listdir(en_path):
                        for fmt in FORMATS:
                            if fmt in f:
                                with open(os.path.join(en_path, f), 'r', encoding='utf-8') as file:
                                    config[cat_key][price_key]["EN"][fmt] = file.read()

    # 写入 JSON 文件
    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(config, f, ensure_ascii=False, indent=4)
    
    print(f"成功！已生成 {OUTPUT_JSON}，快去查看文件内容吧。")

if __name__ == "__main__":
    build_config()

In [1]:
import os
import json

# 配置你的根目录名称
BASE_DIR = "template"
OUTPUT_JSON = "experiment_config.json"

def build_config():
    # 初始化结构
    config = {
        "Apparel": {"budget": {"CN":{}, "EN":{}}, "medium": {"CN":{}, "EN":{}}, "premium": {"CN":{}, "EN":{}}},
        "Cosmetics": {"budget": {"CN":{}, "EN":{}}, "medium": {"CN":{}, "EN":{}}, "premium": {"CN":{}, "EN":{}}},
        "Electronics": {"budget": {"CN":{}, "EN":{}}, "medium": {"CN":{}, "EN":{}}, "premium": {"CN":{}, "EN":{}}}
    }

    print("开始扫描文件夹...")

    # 使用 os.walk 深度遍历所有文件
    for root, dirs, files in os.walk(BASE_DIR):
        for file in files:
            # 只处理 .txt 和 .md 文件
            if not (file.endswith(".txt") or file.endswith(".md")):
                continue

            path = root.replace("\\", "/") # 统一路径分隔符
            
            # 1. 确定品类 (Category)
            cat_key = None
            if "Apparel" in path: cat_key = "Apparel"
            elif "Cosmetics" in path: cat_key = "Cosmetics"
            elif "Electronics" in path: cat_key = "Electronics"
            
            # 2. 确定价位 (Price)
            price_key = None
            if "budget" in path: price_key = "budget"
            elif "medium" in path: price_key = "medium"
            elif "premium" in path: price_key = "premium"
            
            # 3. 确定语言 (Language)
            lang_key = None
            if "核心卖点前置" in path: lang_key = "CN"
            elif "Core Selling Points" in path: lang_key = "EN"
            
            # 4. 确定格式 (Format)
            fmt_key = None
            if "FAQ" in file: fmt_key = "FAQ"
            elif "List" in file: fmt_key = "List"
            elif "Paragraph" in file: fmt_key = "Paragraph"

            # 如果四个关键信息都找到了，就读取文件
            if cat_key and price_key and lang_key and fmt_key:
                file_path = os.path.join(root, file)
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        config[cat_key][price_key][lang_key][fmt_key] = f.read()
                        print(f"已读取: {cat_key} > {price_key} > {lang_key} > {fmt_key}")
                except Exception as e:
                    print(f"读取失败 {file_path}: {e}")

    # 写入 JSON 文件
    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(config, f, ensure_ascii=False, indent=4)
    
    print("-" * 30)
    print(f"大功告成！已生成 {OUTPUT_JSON}")

if __name__ == "__main__":
    build_config()

开始扫描文件夹...
------------------------------
大功告成！已生成 experiment_config.json
